# 01. 단기 메모리(Short-Term Memory)

> Part 02에서 배운 체크포인터 기초를 바탕으로, 한 사용자 세션 안의 대화 이력을 **어떻게 줄이고·삭제하고·요약할지** 다룹니다.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. Part 02의 `MemorySaver` + `thread_id` 패턴을 짧게 복습하고 단기 메모리 예제에 적용할 수 있어요
2. `get_state()` / `get_state_history()`로 대화 스냅샷을 조회하고 디버깅할 수 있어요
3. `trim_messages`로 LLM 호출 직전 컨텍스트 윈도우를 관리할 수 있어요
4. `RemoveMessage`로 체크포인터에 저장된 메시지를 영구적으로 삭제할 수 있어요
5. 대화 요약(Summarization)으로 긴 대화의 핵심 정보를 압축할 수 있어요
6. 개발용 `MemorySaver`와 프로덕션용 `PostgresSaver`의 역할 차이를 설명할 수 있어요

## 사전 지식

- `02_LangGraph_Basics/07-Memory-Checkpointer.ipynb`: checkpointer, `thread_id`, 기본 멀티턴 대화
- `02_LangGraph_Basics/09-State-Management.ipynb`: state 조회와 replay 개념
- Part 06의 Middleware 패턴 (선택)


## 이 노트북의 위치

LLM과 LangGraph의 기본 체크포인터 동작은 `02_LangGraph_Basics/07-Memory-Checkpointer.ipynb`에서 이미 배웠어요. 여기서는 같은 내용을 길게 반복하지 않고, **체크포인터에 쌓이는 대화 이력을 운영 가능한 크기로 관리하는 방법**에 초점을 둡니다.

단기 메모리 관리는 세 가지 질문으로 나눠서 생각하면 쉬워요:

| 질문 | 사용할 도구 | 저장 상태에 미치는 영향 |
|---|---|---|
| 이번 모델 호출에 너무 많은 메시지가 들어가나요? | `trim_messages` | 저장된 원본은 유지, LLM 입력만 줄임 |
| 민감하거나 불필요한 메시지를 완전히 지워야 하나요? | `RemoveMessage` | 체크포인터 상태에서도 삭제 |
| 오래된 대화의 의미는 살리고 토큰은 줄이고 싶나요? | Summarization | 원본 일부를 요약으로 대체 |

> 🔁 **복습 연결**: `MemorySaver`, `thread_id`, `get_state()`의 기본 의미가 헷갈리면 Part 02의 memory/checkpointer 단원을 먼저 확인해요. 이 노트북의 새 목표는 “기억하게 만들기”가 아니라 “기억을 관리하기”입니다.


## 전체 아키텍처

이 노트북에서 다루는 단기 메모리 관리의 전체 흐름이에요:

```{mermaid}
flowchart TD
    subgraph 기본["1. 기본 메모리 챗봇"]
        A[사용자 입력] --> B[챗봇 노드]
        B --> C[MemorySaver]
    end

    subgraph 관리["2. 단기 메모리 관리 전략"]
        D[메시지 누적] --> E{길이 초과?}
        E -->|트리밍| F[최근 N개 유지]
        E -->|삭제| G[영구 제거]
        E -->|요약| H[압축 저장]
    end

    subgraph 프로덕션["3. 프로덕션"]
        I[PostgresSaver]
        J[재시작 후에도 유지]
        I --> J
    end

    C --> D
    F --> 프로덕션
    G --> 프로덕션
    H --> 프로덕션

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    class A input
    class B,D,E process
    class C,I storage
    class F,G,H,J output
```

## 환경 설정

In [ ]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에서 OPENAI_API_KEY 등을 읽어와요
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택사항)
# ---------------------------------------------------
# LangSmith에서 실행 과정을 시각화하고 디버깅할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-07-Short-Term-Memory"

# LangSmith 추적 설정 완료!

---

## 1. MemorySaver로 기본 챗봇 만들기

먼저 Part 02의 `compile(checkpointer=...)` 패턴을 짧게 재사용해 단기 메모리 실험용 챗봇을 준비합니다. 이 섹션은 새 개념이 아니라 이후의 트리밍·삭제·요약 실습을 위한 공통 베이스라인이에요.

> ⚠️ **복습 포인트**: `MemorySaver`는 개발용 인메모리 체크포인터입니다. 프로덕션 저장소 선택은 뒤쪽 `PostgresSaver` 섹션에서 다시 정리합니다.


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model  # V1 모델 초기화
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages  # 메시지 누적 리듀서
from langgraph.checkpoint.memory import MemorySaver  # 인메모리 체크포인터

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → chatbot → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 1.1 thread_id로 멀티턴 대화 테스트

`thread_id`는 Part 02에서 배운 것처럼 대화 세션 식별자예요. 여기서는 기본 개념을 다시 설명하기보다, 이후 메모리 관리 전략을 적용할 **같은 세션/다른 세션 비교 기준**으로 사용합니다.


In [ ]:
from langchain_core.runnables import RunnableConfig

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 체크포인트 라이프사이클

체크포인터가 슈퍼-스텝마다 상태를 저장한다는 점은 Part 02의 핵심 개념이었어요. 이 노트북에서는 그 라이프사이클을 다음처럼 메모리 관리 관점으로 읽습니다.

```mermaid
sequenceDiagram
    participant U as 사용자
    participant G as 그래프
    participant C as 체크포인터
    participant S as 상태 저장소

    U->>G: invoke(messages, config)
    G->>C: thread_id로 상태 복원
    C-->>G: 이전 대화 State
    G->>G: trim / delete / summarize 적용 가능
    G->>C: 슈퍼-스텝 완료, 새 상태 저장
    G-->>U: 응답 반환
```

> 💡 **핵심 정리**: 단기 메모리 전략은 이 저장 주기 안에서 “무엇을 LLM에 보여주고, 무엇을 체크포인터에 남길지”를 결정하는 일입니다.


---

## 2. 상태 조회: get_state() & get_state_history()

`get_state()`와 `get_state_history()`도 Part 02에서 배운 기능이에요. 여기서는 이 기능을 **메모리 관리 전후 상태를 확인하는 디버깅 도구**로 사용합니다.

> 💡 **핵심 정리**: 트리밍은 LLM 입력만 줄이고, `RemoveMessage`와 요약은 저장 상태 자체를 바꿔요. 각 실습 뒤에 상태를 조회하면서 이 차이를 확인해요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 단기 메모리 관리 3가지 전략 비교

이 노트북에서 배울 세 가지 전략을 시각적으로 비교해볼게요.

```mermaid
flowchart LR
    subgraph trim["1. trim_messages"]
        direction TB
        T1["전체 메시지<br>[A, B, C, D, E]"]
        T2["LLM에 전달<br>[D, E]"]
        T3["체크포인터<br>[A, B, C, D, E]"]
        T1 --> T2
        T1 --> T3
    end

    subgraph remove["2. RemoveMessage"]
        direction TB
        R1["전체 메시지<br>[A, B, C, D, E]"]
        R2["삭제 후<br>[C, D, E]"]
        R3["체크포인터<br>[C, D, E]"]
        R1 --> R2
        R2 --> R3
    end

    subgraph summary["3. 대화 요약"]
        direction TB
        S1["전체 메시지<br>[A, B, C, D, E]"]
        S2["요약 + 최근<br>[요약문, D, E]"]
        S3["체크포인터<br>[요약문, D, E]"]
        S1 --> S2
        S2 --> S3
    end

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    class T1,R1,S1 input
    class T2,R2,S2 process
    class T3,R3,S3 storage
```

| 전략 | 체크포인터 저장 | 정보 손실 | 추가 비용 |
|------|:-:|:-:|:-:|
| **trim_messages** | 전체 유지 | 일시적 | 없음 |
| **RemoveMessage** | 삭제됨 | 영구적 | 없음 |
| **대화 요약** | 요약 교체 | 최소화 | LLM 호출 |

> 🔑 **핵심 개념**: 세 가지 전략은 **체크포인터에 남는 데이터**가 핵심 차이예요. 트리밍은 체크포인터를 건드리지 않고, RemoveMessage는 영구 삭제하고, 요약은 압축하여 대체해요.

---

## 3. 메시지 트리밍(trim_messages)

대화가 길어지면 LLM의 **컨텍스트 윈도우(Context Window)** 한도에 도달해 에러가 발생하거나 비용이 급증해요. `trim_messages`는 LLM 호출 **직전**에 메시지를 잘라내서 토큰 수를 제어해요.

**트리밍의 특징:**
- 잘라낸 메시지는 LLM에 전달되지 않지만, 체크포인터에는 여전히 **저장**돼요
- 즉, 트리밍은 **일시적** 필터링이지 **영구 삭제**가 아니에요

> ⚠️ **자주 하는 실수**: `token_counter=len`을 사용하면 메시지 **개수**를 토큰으로 계산해서 트리밍이 엉뚱하게 작동해요. 반드시 실제 토큰을 세는 함수나 모델 객체를 전달해야 해요.

```python
# 잘못된 사용: 메시지 개수를 토큰으로 잘못 계산
trim_messages(messages, max_tokens=100, token_counter=len)   # 위험!

# 올바른 사용 1: LLM 모델 객체 전달 (정확)
trim_messages(messages, max_tokens=100, token_counter=llm)

# 올바른 사용 2: 근사 카운터 함수 전달 (API 호출 없이 빠름)
def count_tokens_approx(messages):
    return sum(len(m.content) // 4 for m in messages if hasattr(m, 'content'))
```

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain_core.messages import trim_messages

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트용 긴 대화 기록 준비
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3.1 그래프에 트리밍 적용

실제 그래프에서 트리밍을 적용할 때는 **에이전트 노드 내부**에서 LLM 호출 직전에 트리밍해요. 체크포인터에는 전체 기록이 저장되지만, LLM에는 트리밍된 메시지만 전달돼요.

> 💡 **실무 팁**: `max_tokens`를 모델의 컨텍스트 윈도우 크기의 80% 정도로 설정하면 여유 공간을 남기면서 안정적으로 운용할 수 있어요. gpt-4o-mini는 128K 토큰이므로 100K 정도로 설정하는 게 좋아요.

In [ ]:
from langchain_core.messages import trim_messages

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → chatbot → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 4. RemoveMessage로 메시지 영구 삭제

트리밍은 일시적 필터링이지만, `RemoveMessage`는 체크포인터의 상태에서 메시지를 **영구적으로 제거**해요. 두 방식의 차이를 명확히 이해해야 해요.

| 구분 | trim_messages | RemoveMessage |
|------|--------------|---------------|
| 저장 상태 | 체크포인터에 유지 | 체크포인터에서도 삭제 |
| 복구 가능 | 가능 | 불가능 |
| 사용 시점 | LLM 호출 직전 | 상태 직접 수정 시 |
| 주요 용도 | 컨텍스트 윈도우 관리 | 민감 정보 삭제, 저장 공간 절약 |

> 🔑 **핵심 개념**: `RemoveMessage(id=msg_id)`는 add_messages 리듀서에게 "이 id의 메시지를 삭제하라"는 신호예요. 각 메시지는 생성 시 자동으로 고유한 `id`가 부여돼요.

In [ ]:
from langchain_core.messages import RemoveMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 4.1 그래프 내부에서 자동 삭제 패턴

매번 수동으로 삭제하는 대신, **그래프 노드**에서 자동으로 오래된 메시지를 삭제할 수 있어요. 이 패턴은 메시지 개수가 일정 수를 넘으면 자동으로 정리해요.

> 💡 **실무 팁**: 이 패턴은 저장 비용이 중요한 프로덕션 환경에서 유용해요. 반면 대화 이력 분석이나 감사(Audit) 목적으로 모든 메시지를 보존해야 한다면 사용하지 말아야 해요.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools/delete_messages → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 5. 대화 요약(Summarization)으로 컨텍스트 압축

트리밍과 삭제는 **정보 손실**이 발생해요. 대화 요약은 오래된 메시지들을 LLM으로 압축해서 핵심 정보는 보존하면서 토큰을 줄이는 방법이에요.

> 💡 **핵심 정리**: 요약 전략은 마치 **회의록 작성자**와 같아요. 1시간짜리 회의 내용을 녹음 전체 들려줄 수도 있지만(비효율), 핵심만 정리한 회의록 한 페이지를 주는 게 훨씬 낫죠. 요약 패턴은 "오래된 대화 전체"를 "요약 한 문단"으로 바꿔서 비용도 줄이고 정보도 보존해요.

**요약 전략 흐름:**
```
[오래된 메시지들] → LLM 요약 → [한 줄 요약 시스템 메시지]
                          ↓
               원본 메시지는 삭제 (RemoveMessage)
```

> ⚠️ **자주 하는 실수**: 요약도 LLM 호출이므로 **추가 비용**이 발생해요. 너무 자주 요약하면 오히려 비용이 더 들 수 있어요. 보통 메시지가 10-20개를 넘으면 요약을 트리거하는 게 좋아요.

> 💡 **실무 팁**: 요약에는 메인 모델보다 **작고 빠른 모델**(예: gpt-4o-mini)을 사용하면 비용을 절약할 수 있어요. 요약은 단순 압축 작업이라 작은 모델로도 충분해요.

In [ ]:
from typing_extensions import TypedDict

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → summarize → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 6. 프로덕션 체크포인터: PostgresSaver

`MemorySaver`는 개발/테스트 전용이에요. 실제 서비스에서는 **영구 저장소**가 필요해요.

`PostgresSaver`는 가장 대표적인 프로덕션 체크포인터예요:
- 서버 재시작 후에도 대화 이력이 유지돼요
- 여러 서버 인스턴스에서 동일한 상태를 공유할 수 있어요
- ACID 트랜잭션으로 데이터 무결성을 보장해요

> ⚠️ **자주 하는 실수**: `PostgresSaver.from_conn_string()` 사용 시 반드시 `with` 블록(컨텍스트 매니저)을 사용해야 해요. 그렇지 않으면 연결이 제대로 닫히지 않아 **연결 누수**가 발생해요.

### Docker로 PostgreSQL 빠르게 설정하기

```bash
# PostgreSQL 컨테이너 실행 (한 번만 실행)
docker run --name langgraph_db \
    -e POSTGRES_PASSWORD=postgres \
    -e POSTGRES_DB=langgraph_db \
    -p 5432:5432 \
    -d postgres:15

# 필요한 패키지 설치
pip install langgraph-checkpoint-postgres
```

> 💡 **실무 팁**: Docker를 사용하면 로컬 환경에서도 프로덕션과 동일한 체크포인터로 테스트할 수 있어요. 강의 환경에서는 시간이 없다면 코드 설명만 하고 실행은 선택사항으로 안내해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 7. 실습: 나만의 메모리 전략 구현

배운 내용을 종합해서 직접 구현해볼게요!

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 하이브리드 메모리 전략 구현
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **기초 복습**: `MemorySaver`, `thread_id`, `get_state()`는 Part 02에서 배운 체크포인터 패턴이며, 여기서는 단기 메모리 실험의 베이스라인으로 사용했어요
- **trim_messages**: LLM 호출 직전에 메시지를 필터링해요. 체크포인터에는 원본이 유지됩니다
- **RemoveMessage**: 체크포인터의 상태에서 특정 메시지를 영구적으로 삭제해요
- **대화 요약**: 오래된 메시지를 LLM으로 압축해 정보 손실을 줄이면서 토큰을 절약해요
- **PostgresSaver**: 프로덕션용 영구 체크포인터이며, 개발용 `MemorySaver`와 역할이 달라요

### 단기 메모리 관리 전략 비교

| 전략 | 체크포인터 저장 | 정보 손실 | 추가 비용 | 주요 용도 |
|------|--------------|----------|----------|----------|
| trim_messages | 유지됨 | 있음 (일시적) | 없음 | 컨텍스트 윈도우 관리 |
| RemoveMessage | 삭제됨 | 있음 (영구) | 없음 | 민감 정보 삭제 |
| 대화 요약 | 요약으로 대체 | 최소화 | LLM 호출 | 긴 대화 압축 |


## 다음 노트북 예고

다음 `02-Long-Term-Memory.ipynb`에서는 **장기 메모리(Long-Term Memory)**를 배워요. `Store API`를 사용해 `thread_id`를 넘어 **사용자 간 공유 가능한** 영구 메모리를 구현하고, PostgresSaver로 세션이 재시작되어도 대화 이력을 완전히 복구하는 방법을 다뤄요.